Imports & Cache Setup

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import faiss
import json
from sentence_transformers import SentenceTransformer, InputExample
import chromadb
from chromadb.config import Settings
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Make cache folder
os.makedirs("cache", exist_ok=True)

# Quick check that FAISS and Chroma are installed
print("FAISS version:", faiss.__version__)
print("Chroma path:", chromadb.__file__)

/Users/tobiloba/miniforge3/envs/rag310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FAISS version: 1.7.4
Chroma path: /Users/tobiloba/miniforge3/envs/rag310/lib/python3.10/site-packages/chromadb/__init__.py


Load the Dataset

In [2]:
# Replace this path with your uploaded file path
path = "labelled_newscatcher_dataset.csv"

pdf = pd.read_csv(path, on_bad_lines='skip', encoding='utf-8', engine='python')
pdf.head()



,topic;link;domain;published_date;title;lang
SCIENCE;https://www.eurekalert.org/pub_releases/2020-08/dbnl-acl080620.php;eurekalert.org;2020-08-06 13:59:45;A closer look at water-splitting's solar fuel potential;en,
SCIENCE;https://www.pulse.ng/news/world/an-irresistible-scent-makes-locusts-swarm-study-finds/jy784jw;pulse.ng;2020-08-12 15:14:19;An irresistible scent makes locusts swarm,study finds;en
SCIENCE;https://www.express.co.uk/news/science/1322607/artificial-intelligence-warning-machine-learning-algorithm-social-media-data;express.co.uk;2020-08-13 21:01:00;Artificial intelligence warning: AI will know us better than we know ourselves;en,None
SCIENCE;https://www.ndtv.com/world-news/glaciers-could-have-sculpted-mars-valleys-study-2273648;ndtv.com;2020-08-03 22:18:26;Glaciers Could Have Sculpted Mars Valleys: Study;en,None
SCIENCE;https://www.thesun.ie/tech/5742187/perseid-meteor-shower-tonight-time-uk-see/;thesun.ie;2020-08-12 19:54:36;Perseid meteor shower 2020: What time and how to see the huge bright FIREBALLS over UK again tonight;en,None
SCIENCE;https://interestingengineering.com/nasa-releases-in-depth-map-of-beirut-explosion-damage;interestingengineering.com;2020-08-08 11:05:45;NASA Releases In-Depth Map of Beirut Explosion Damage;en,None


Add an Identifier Columnn

In [6]:
pdf["id"] = range(1, len(pdf) + 1)
pdf.head()

,id,topic,link,domain,published_date,title,lang
0,1,SCIENCE,https://www.eurekalert.org/pub_releases/2020-0...,eurekalert.org,2020-08-06 13:59:45,A closer look at water-splitting's solar fuel ...,en
1,2,SCIENCE,https://www.pulse.ng/news/world/an-irresistibl...,pulse.ng,2020-08-12 15:14:19,"An irresistible scent makes locusts swarm, stu...",en
2,3,SCIENCE,https://www.express.co.uk/news/science/1322607...,express.co.uk,2020-08-13 21:01:00,Artificial intelligence warning: AI will know ...,en
3,4,SCIENCE,https://www.ndtv.com/world-news/glaciers-could...,ndtv.com,2020-08-03 22:18:26,Glaciers Could Have Sculpted Mars Valleys: Study,en
4,5,SCIENCE,https://www.thesun.ie/tech/5742187/perseid-met...,thesun.ie,2020-08-12 19:54:36,Perseid meteor shower 2020: What time and how ...,en


Inspect the Data

In [7]:
pdf.info()
pdf.isnull().sum()
pdf.head(3)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108774 entries, 0 to 108773
Data columns (total 7 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   id              108774 non-null  int64 
 1   topic           108774 non-null  object
 2   link            108774 non-null  object
 3   domain          108774 non-null  object
 4   published_date  108774 non-null  object
 5   title           108774 non-null  object
 6   lang            108774 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.8+ MB


,id,topic,link,domain,published_date,title,lang
0,1,SCIENCE,https://www.eurekalert.org/pub_releases/2020-0...,eurekalert.org,2020-08-06 13:59:45,A closer look at water-splitting's solar fuel ...,en
1,2,SCIENCE,https://www.pulse.ng/news/world/an-irresistibl...,pulse.ng,2020-08-12 15:14:19,"An irresistible scent makes locusts swarm, stu...",en
2,3,SCIENCE,https://www.express.co.uk/news/science/1322607...,express.co.uk,2020-08-13 21:01:00,Artificial intelligence warning: AI will know ...,en


Create a Subset for Faster Processing 

In [5]:
pdf_subset = pdf.head(1000).copy()
pdf_subset.head()

,id,topic,link,domain,published_date,title,lang
0,1,SCIENCE,https://www.eurekalert.org/pub_releases/2020-0...,eurekalert.org,2020-08-06 13:59:45,A closer look at water-splitting's solar fuel ...,en
1,2,SCIENCE,https://www.pulse.ng/news/world/an-irresistibl...,pulse.ng,2020-08-12 15:14:19,"An irresistible scent makes locusts swarm, stu...",en
2,3,SCIENCE,https://www.express.co.uk/news/science/1322607...,express.co.uk,2020-08-13 21:01:00,Artificial intelligence warning: AI will know ...,en
3,4,SCIENCE,https://www.ndtv.com/world-news/glaciers-could...,ndtv.com,2020-08-03 22:18:26,Glaciers Could Have Sculpted Mars Valleys: Study,en
4,5,SCIENCE,https://www.thesun.ie/tech/5742187/perseid-met...,thesun.ie,2020-08-12 19:54:36,Perseid meteor shower 2020: What time and how ...,en


Exercise 2: Vectorization with Sentence Transformers

In [8]:
pdf_subset = pdf.iloc[:1000].copy()

In [9]:
def example_create_fn(doc1: pd.Series) -> InputExample:
    """
    Helper function that outputs a sentence_transformer guid, label, and text.
    """
    return ...  # Format and return the InputExample.

In [10]:
faiss_train_examples = pdf_subset.apply(lambda x: example_create_fn(x["title"]), axis=1).tolist()
faiss_train_examples[:10]

[Ellipsis,
 Ellipsis,
 Ellipsis,
 Ellipsis,
 Ellipsis,
 Ellipsis,
 Ellipsis,
 Ellipsis,
 Ellipsis,
 Ellipsis]

In [ ]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name, device="cpu")  # use "cuda" if you have GPU

In [ ]:

# 3) Load a solid sentence embedding model
model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name, device="cpu")  # use "cuda" if you have GPU

# (Optional) sanity: embedding dimensionality
dim = model.get_sentence_embedding_dimension()
print(f"Embedding dimension reported by model: {dim}")

# 4) Prepare a plain list of strings to actually embed
titles_list = pdf_subset["title"].astype(str).tolist()

# 5) Generate embeddings (batched, deterministic)
faiss_title_embedding = model.encode(
    titles_list,
    batch_size=64,
    convert_to_numpy=True,
    normalize_embeddings=False,  # we'll handle normalization later for cosine/IP
    show_progress_bar=True
).astype(np.float32)  # FAISS prefers float32

# 6) Quick checks
print("Embeddings shape:", faiss_title_embedding.shape)    # -> (num_rows, dim)
print("First vector (first 8 vals):", faiss_title_embedding[0][:8])

# Keep handy variables for the next exercise
embed_matrix = faiss_title_embedding  # (n, d)
id_index = pdf_subset.index.to_numpy(dtype=np.int64)       # stable row ids